In [1]:
import os
import pandas as pd
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm


In [2]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# import torch
# from torch.utils.data import DataLoader, TensorDataset
# import torch.nn as nn

# from tensorflow.keras.models import Sequential, load_model, Model
# from tensorflow.keras.layers import LSTM, GRU, Bidirectional, Dense, Input, Concatenate, Dropout
# from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


In [4]:
# 讀取用戶上傳的CSV檔案
file_path = 'breeding_logs.csv'

# 欄位名稱
columns = [
    'id', 'contract_id', 'chicken_import_id', 'building_number', 'date', 'age', 
    'disuse', 'breeding_rate', 'am_avg_weight', 'pm_avg_weight', 'creator', 
    'last_editor', 'modified_log', 'created_at', 'updated_at'
]

# 讀取CSV檔案，並加入指定的欄位名稱
df = pd.read_csv(file_path, header=None, names=columns)

# 顯示前幾筆資料來確認讀取成功
df

,id,contract_id,chicken_import_id,building_number,date,age,disuse,breeding_rate,am_avg_weight,pm_avg_weight,creator,last_editor,modified_log,created_at,updated_at
0,35,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11212102,A棟上層,2023-12-11,0,0,1.02,\N,33,\N,\N,\N,2024-06-14 03:36:50,2024-07-13 07:04:17
1,36,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11212102,A棟上層,2023-12-12,1,0,1.02,34,35,\N,\N,\N,2024-06-14 03:36:50,2024-07-13 07:04:17
2,37,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11212102,A棟上層,2023-12-13,2,0,1.02,36,38,\N,\N,\N,2024-06-14 03:36:50,2024-07-13 07:04:17
3,38,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11212102,A棟上層,2023-12-14,3,0,1.02,42,45,\N,\N,\N,2024-06-14 03:36:50,2024-07-13 07:04:17
4,39,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11212102,A棟上層,2023-12-15,4,0,1.02,56,73,\N,\N,\N,2024-06-14 03:36:50,2024-07-13 07:04:17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1709,1760,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11303103,上層,2024-03-27,11,0,1.02,285,319,\N,\N,\N,2024-07-12 04:25:23,2024-07-12 04:25:23
1710,1761,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11303103,上層,2024-03-28,12,0,1.02,338,374,\N,\N,\N,2024-07-12 04:25:23,2024-07-12 04:25:23
1711,1762,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11303103,上層,2024-03-29,13,0,1.02,389,429,\N,\N,\N,2024-07-12 04:25:23,2024-07-12 04:25:23
1712,1763,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11303103,上層,2024-03-30,14,0,1.02,448,493,\N,\N,\N,2024-07-12 04:25:23,2024-07-12 04:25:23


In [5]:
# Dropping the unnecessary columns as specified by the user
columns_to_drop = ['id', 'creator', 'last_editor', 'modified_log', 'created_at', 'updated_at', 'date', 'disuse', 'breeding_rate'] # 'contract_id', 
df = df.drop(columns=columns_to_drop)
df

,contract_id,chicken_import_id,building_number,age,am_avg_weight,pm_avg_weight
0,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11212102,A棟上層,0,\N,33
1,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11212102,A棟上層,1,34,35
2,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11212102,A棟上層,2,36,38
3,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11212102,A棟上層,3,42,45
4,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11212102,A棟上層,4,56,73
...,...,...,...,...,...,...
1709,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11303103,上層,11,285,319
1710,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11303103,上層,12,338,374
1711,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11303103,上層,13,389,429
1712,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11303103,上層,14,448,493


In [6]:
# 查看缺失值，篩選出 'am_avg_weight' 及 'pm_avg_weight' 皆為空值或0的行。
filtered_df = df[
    ((df['am_avg_weight'] == "\\N") | (df['am_avg_weight'] == "0")) &
    ((df['pm_avg_weight'] == "\\N") | (df['pm_avg_weight'] == "0"))
]

filtered_df

,contract_id,chicken_import_id,building_number,age,am_avg_weight,pm_avg_weight
130,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11302101,B棟上層,32,\N,\N
131,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11302101,B棟上層,33,\N,\N
132,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11302101,B棟上層,34,\N,\N
133,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11302101,B棟上層,35,\N,\N
1084,eef47f04-01e7-4b49-8827-e231694ea345,G11303104,上層,34,\N,\N
...,...,...,...,...,...,...
1388,ac9f83ba-6c2f-4e40-ae65-e4593ddab5bf,G11304103,三場,31,\N,\N
1389,ac9f83ba-6c2f-4e40-ae65-e4593ddab5bf,G11304103,三場,32,\N,\N
1390,ac9f83ba-6c2f-4e40-ae65-e4593ddab5bf,G11304103,三場,33,\N,\N
1391,ac9f83ba-6c2f-4e40-ae65-e4593ddab5bf,G11304103,三場,34,\N,\N


In [7]:
# 並且依據chicken_import_id及building_number群組，計算各別所缺失的資料數量。
#   => 需要補齊資料，若該雞隻的資料缺失太多，則丟棄。

# 按 'age' 排序並選取每個 'chicken_import_id' 和 'building_number' 組合的最後一筆資料
last_age_df = filtered_df.sort_values('age').drop_duplicates(subset=['chicken_import_id', 'building_number'], keep='last')

# 新增一個欄位來計算每個 'chicken_import_id' 和 'building_number' 組合的資料數量
last_age_df['data_count'] = filtered_df.groupby(['chicken_import_id', 'building_number'])['age'].transform('count')

# 顯示結果
# last_age_df.drop(columns=['am_avg_weight',	'pm_avg_weight', 'age']).reset_index(drop=True)
last_age_df.drop(columns=['am_avg_weight',	'pm_avg_weight'])

,contract_id,chicken_import_id,building_number,age,data_count
1295,1163b6f9-48f9-4ec6-8aba-927c6e03be46,A11305106,下層,8,1
1220,eef47f04-01e7-4b49-8827-e231694ea345,G11303104,下層,35,2
1085,eef47f04-01e7-4b49-8827-e231694ea345,G11303104,上層,35,2
133,f5db36da-e119-40db-923f-0e06f6ab5a2c,G11302101,B棟上層,35,4
1356,1163b6f9-48f9-4ec6-8aba-927c6e03be46,A11301101,上層,35,36
1392,ac9f83ba-6c2f-4e40-ae65-e4593ddab5bf,G11304103,三場,35,36


<ol> 計算這隻雞總共缺幾次。
    <li> 針對 filtered_df（所有缺失），計算每隻雞 (chicken_import_id, building_number) 的 缺失筆數。 </li>
    <li> 表格裡的 data_count 就是「這隻雞在所有日齡裡，一共缺失多少次」。 </li>
</ol>

<h3> 遇到缺失值的處理方式 </h3> 
<ol>
    <li> 先去系統查看是否真的缺少均重資料。 </li>
    <li> 從表中我們可以看到有些資料缺失太多，幾乎是沒有任何重量資料，應當刪除。 </li>
        <p> A11301101	上層 </p>
        <p> G11304103	三場 </p>
    <li> 其餘資料則找是否有相同的批號（可以不同棟舍）來填補缺失值。 </li>
    
</ol>

In [8]:
# 一次刪除多筆特定的 chicken_import_id 和 building_number 組合
# 構建要刪除的條件
delete_conditions = [
    ('A11301101', '上層'),
    ('G11304103', '三場')
]

# 將條件轉換為 DataFrame 以便進行匹配
conditions_df = pd.DataFrame(delete_conditions, columns=['chicken_import_id', 'building_number'])

# 使用 merge 來進行篩選，刪除符合條件的資料
df = df.merge(conditions_df, on=['chicken_import_id', 'building_number'], how='outer', indicator=True)
df = df[df['_merge'] == 'left_only'].drop(columns=['_merge'])

df

,contract_id,chicken_import_id,building_number,age,am_avg_weight,pm_avg_weight
36,1163b6f9-48f9-4ec6-8aba-927c6e03be46,A11303103,下層,0,\N,41
37,1163b6f9-48f9-4ec6-8aba-927c6e03be46,A11303103,下層,1,52,58
38,1163b6f9-48f9-4ec6-8aba-927c6e03be46,A11303103,下層,2,64,72
39,1163b6f9-48f9-4ec6-8aba-927c6e03be46,A11303103,下層,3,84,90
40,1163b6f9-48f9-4ec6-8aba-927c6e03be46,A11303103,下層,4,90,88
...,...,...,...,...,...,...
1709,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11305103,上層,29,1617,1684
1710,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11305103,上層,30,1725,1770
1711,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11305103,上層,31,1749,1834
1712,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11305103,上層,32,1777,1882


In [9]:
# 新增'avg_weight'欄位的邏輯
df['avg_weight'] = df.apply(
    lambda row: row['pm_avg_weight'] if pd.notnull(row['pm_avg_weight']) and row['pm_avg_weight'] != "\\N" and row['pm_avg_weight'] != "0"
    else row['am_avg_weight'] if pd.notnull(row['am_avg_weight']) and row['am_avg_weight'] != "\\N" and row['am_avg_weight'] != "0"
    else None, axis=1
)

# 將'avg_weight'轉換為數值型態
# errors='coerce' 的意思是，如果遇到無法轉換為數字的值（例如字母或特殊符號），Pandas 會將該值轉換為 NaN（空值），而不會拋出錯誤。
df['avg_weight'] = pd.to_numeric(df['avg_weight'], errors='coerce')

# 刪除'am_avg_weight'和'pm_avg_weight'欄位
df = df.drop(columns=['am_avg_weight', 'pm_avg_weight'])

df

,contract_id,chicken_import_id,building_number,age,avg_weight
36,1163b6f9-48f9-4ec6-8aba-927c6e03be46,A11303103,下層,0,41.0
37,1163b6f9-48f9-4ec6-8aba-927c6e03be46,A11303103,下層,1,58.0
38,1163b6f9-48f9-4ec6-8aba-927c6e03be46,A11303103,下層,2,72.0
39,1163b6f9-48f9-4ec6-8aba-927c6e03be46,A11303103,下層,3,90.0
40,1163b6f9-48f9-4ec6-8aba-927c6e03be46,A11303103,下層,4,88.0
...,...,...,...,...,...
1709,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11305103,上層,29,1684.0
1710,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11305103,上層,30,1770.0
1711,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11305103,上層,31,1834.0
1712,62c0b26d-85a0-479e-8607-3e7eca0a4fe2,G11305103,上層,32,1882.0


<h3> 逐筆處理缺失值 </h3>

<p style='color:blue'>◆ chicken_import_id為A11305106的資料 </p>

In [10]:
#  chicken_import_id為A11305106的資料
# 搜尋 'chicken_import_id' 為 'A11305106' 且 'age' 為 8 的資料
search_result = df[(df['chicken_import_id'] == 'A11305106') & (df['avg_weight'].isnull())]
search_result

,contract_id,chicken_import_id,building_number,age,avg_weight
78,1163b6f9-48f9-4ec6-8aba-927c6e03be46,A11305106,下層,8,NaN
